In [13]:
import warnings
warnings.filterwarnings("ignore")

In [1]:
   import numpy as np
   import pandas as pd
   import sklearn
   print("pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

pandas 3.1.1 | scikit-learn 1.9.1


In [1]:
from pathlib import Path

root = Path("dataset")
for p in sorted(root.rglob("*"))[:40]:
    kind = "folder" if p.is_dir() else f"{p.stat().st_size/1e6:.1f} MB"
    print(p, "|", kind)

dataset\Life labels | folder
dataset\Life labels\CALB_labels.json | 0.0 MB
dataset\Life labels\CALCE_labels.json | 0.0 MB
dataset\Life labels\HNEI_labels.json | 0.0 MB
dataset\Life labels\HUST_labels.json | 0.0 MB
dataset\Life labels\ISU-ILCC_labels.json | 0.0 MB
dataset\Life labels\MATR_labels.json | 0.0 MB
dataset\Life labels\NA-coin_labels.json | 0.0 MB
dataset\Life labels\NA-ion_labels.json | 0.0 MB
dataset\Life labels\RWTH_labels.json | 0.0 MB
dataset\Life labels\SNL_labels.json | 0.0 MB
dataset\Life labels\Stanford_labels.json | 0.0 MB
dataset\Life labels\Tongji_labels.json | 0.0 MB
dataset\Life labels\total_MICH_labels.json | 0.0 MB
dataset\Life labels\UL-PUR_labels.json | 0.0 MB
dataset\Life labels\XJTU_labels.json | 0.0 MB
dataset\Life labels\ZN-coin_labels.json | 0.0 MB
dataset\SNL | folder
dataset\SNL\SNL_18650_LFP_15C_0-100_0.5-2C_b.pkl | 19.3 MB
dataset\SNL\SNL_18650_LFP_25C_0-100_0.5-0.5C_a.pkl | 23.5 MB
dataset\SNL\SNL_18650_LFP_25C_0-100_0.5-1C_a.pkl | 20.8 MB
dataset\S

In [2]:
import json
from pathlib import Path

with open(Path("dataset/Life labels/SNL_labels.json")) as f:
    labels = json.load(f)

print(type(labels), "| number of entries:", len(labels))
for k in list(labels)[:5]:
    print(k, "->", labels[k])

<class 'dict'> | number of entries: 52
SNL_18650_NCA_15C_0-100_0.5-1C_a.pkl -> 331
SNL_18650_NCA_25C_20-80_0.5-0.5C_d.pkl -> 840
SNL_18650_NMC_15C_0-100_0.5-1C_b.pkl -> 143
SNL_18650_LFP_35C_0-100_0.5-1C_b.pkl -> 3162
SNL_18650_NCA_25C_0-100_0.5-1C_c.pkl -> 333


In [3]:
import pickle

path = Path("dataset/SNL/SNL_18650_NCA_15C_0-100_0.5-1C_a.pkl")
try:
    with open(path, "rb") as f:
        cell = pickle.load(f)
    print(type(cell))
    print([a for a in dir(cell) if not a.startswith("_")][:30])
except Exception as e:
    print("ERROR:", type(e).__name__, e)

<class 'dict'>
['clear', 'copy', 'fromkeys', 'get', 'items', 'keys', 'pop', 'popitem', 'setdefault', 'update', 'values']


In [4]:
import pandas as pd

df = pd.DataFrame({"cell": list(labels), "life": list(labels.values())})
parts = df["cell"].str.split("_")
df["chemistry"] = parts.str[2]
df["temp"] = parts.str[3]
df["protocol"] = parts.str[4] + "_" + parts.str[5]

print(df.groupby(["chemistry", "temp"])["life"].agg(["count", "min", "median", "max"]))

                count   min  median   max
chemistry temp                           
LFP       25C       4  2325  3020.5  3197
          35C       5  2651  3308.0  3842
NCA       15C       4   226   290.5   632
          25C      12   180   356.5   976
          35C       6   229   366.0   444
NMC       15C       4   143   145.0   152
          25C      11   333   428.0   602
          35C       6   500   579.0   639


In [5]:
print("keys:", list(cell.keys()))
for k, v in cell.items():
    if isinstance(v, (list, tuple)):
        print(k, "-> list of length", len(v))
    elif isinstance(v, dict):
        print(k, "-> dict with keys", list(v.keys())[:8])
    else:
        print(k, "->", type(v).__name__, str(v)[:80])

keys: ['cell_id', 'cycle_data', 'form_factor', 'anode_material', 'cathode_material', 'electrolyte_material', 'nominal_capacity_in_Ah', 'depth_of_charge', 'depth_of_discharge', 'already_spent_cycles', 'max_voltage_limit_in_V', 'min_voltage_limit_in_V', 'max_current_limit_in_A', 'min_current_limit_in_A', 'reference', 'description', 'charge_protocol', 'discharge_protocol', 'SOC_interval']
cell_id -> str SNL_18650_NCA_15C_0-100_0.5-1C_a
cycle_data -> list of length 522
form_factor -> str cylindrical_18650
anode_material -> str graphite
cathode_material -> str NCA
electrolyte_material -> NoneType None
nominal_capacity_in_Ah -> float 2.8800000000000003
depth_of_charge -> float 1.0
depth_of_discharge -> float 1.0
already_spent_cycles -> int 0
max_voltage_limit_in_V -> float 4.2
min_voltage_limit_in_V -> float 2.5
max_current_limit_in_A -> float 6.0
min_current_limit_in_A -> NoneType None
reference -> NoneType None
description -> NoneType None
charge_protocol -> list of length 1
discharge_prot

In [6]:
c0 = cell["cycle_data"][0]
print(type(c0))
if isinstance(c0, dict):
    for k, v in c0.items():
        if isinstance(v, (list, tuple)):
            print(k, "-> list of length", len(v), "| first values:", v[:3])
        else:
            print(k, "->", type(v).__name__, str(v)[:60])

<class 'dict'>
cycle_number -> int 1
current_in_A -> list of length 1252 | first values: [0.0, 0.0, 0.0]
voltage_in_V -> list of length 1252 | first values: [3.337, 3.338, 3.337]
charge_capacity_in_Ah -> list of length 1252 | first values: [0.0, 0.0, 0.0]
discharge_capacity_in_Ah -> list of length 1252 | first values: [0.0, 0.0, 0.0]
time_in_s -> list of length 1252 | first values: [10.007, 20.022, 30.037]
temperature_in_C -> list of length 1252 | first values: [15.609, 15.471, 15.548]
internal_resistance_in_ohm -> NoneType None


In [7]:
import numpy as np

N = 100  # number of early cycles we keep
rows = []
for name, life in labels.items():
    with open(Path("dataset/SNL") / name, "rb") as f:
        c = pickle.load(f)
    cycles = c["cycle_data"]
    caps = []
    for cy in cycles[:N]:
        d = cy["discharge_capacity_in_Ah"]
        caps.append(max(d) if len(d) > 0 else np.nan)
    rows.append({"cell": name, "life": life,
                 "n_cycles_in_file": len(cycles), "caps": caps})
    del c  # free memory

feat = pd.DataFrame(rows)
feat["n_caps"] = feat["caps"].apply(len)
feat["cap_c1"] = feat["caps"].apply(lambda x: x[0])
feat["cap_c10"] = feat["caps"].apply(lambda x: x[9] if len(x) > 9 else np.nan)
feat["cap_c100"] = feat["caps"].apply(lambda x: x[-1])

print(feat[["cell", "life", "n_cycles_in_file", "n_caps",
            "cap_c1", "cap_c10", "cap_c100"]].head(10))
print("cells with fewer than", N, "cycles:", (feat["n_caps"] < N).sum())
print("cells with missing values:", feat["caps"].apply(lambda x: np.isnan(x).any()).sum())

                                     cell  life  n_cycles_in_file  n_caps  \
0    SNL_18650_NCA_15C_0-100_0.5-1C_a.pkl   331               522     100   
1  SNL_18650_NCA_25C_20-80_0.5-0.5C_d.pkl   840              1509     100   
2    SNL_18650_NMC_15C_0-100_0.5-1C_b.pkl   143               513     100   
3    SNL_18650_LFP_35C_0-100_0.5-1C_b.pkl  3162              3342     100   
4    SNL_18650_NCA_25C_0-100_0.5-1C_c.pkl   333               522     100   
5    SNL_18650_NCA_35C_0-100_0.5-1C_a.pkl   395               515     100   
6    SNL_18650_NCA_25C_0-100_0.5-2C_a.pkl   335               786     100   
7    SNL_18650_NMC_35C_0-100_0.5-2C_a.pkl   639               786     100   
8  SNL_18650_NMC_25C_0-100_0.5-0.5C_b.pkl   506               679     100   
9    SNL_18650_NCA_15C_0-100_0.5-2C_a.pkl   226               916     100   

   cap_c1  cap_c10  cap_c100  
0   2.911    2.848     2.615  
1   1.979    1.972     1.924  
2   2.536    2.552     2.284  
3   1.046    1.037     1.027

In [8]:
parts = feat["cell"].str.split("_")
feat["chemistry"] = parts.str[2]
feat["temp"] = parts.str[3].str.replace("C", "").astype(int)

rows = []
for _, r in feat.iterrows():
    caps = np.array(r["caps"])
    for k in [20, 40, 60, 80, 100]:
        c = caps[:k] / caps[0]            # capacity retention
        slope = np.polyfit(np.arange(1, k + 1), c, 1)[0]
        rows.append({"cell": r["cell"], "chemistry": r["chemistry"],
                     "temp": r["temp"], "budget": k,
                     "cap_first": caps[0], "retention": c[-1],
                     "slope": slope, "life": r["life"],
                     "log_life": np.log10(r["life"])})

table = pd.DataFrame(rows)
print(table.shape)
print(table.head(10))
print(table.groupby("chemistry")[["retention", "log_life"]].mean())

(260, 9)
                                     cell chemistry  temp  budget  cap_first  \
0    SNL_18650_NCA_15C_0-100_0.5-1C_a.pkl       NCA    15      20      2.911   
1    SNL_18650_NCA_15C_0-100_0.5-1C_a.pkl       NCA    15      40      2.911   
2    SNL_18650_NCA_15C_0-100_0.5-1C_a.pkl       NCA    15      60      2.911   
3    SNL_18650_NCA_15C_0-100_0.5-1C_a.pkl       NCA    15      80      2.911   
4    SNL_18650_NCA_15C_0-100_0.5-1C_a.pkl       NCA    15     100      2.911   
5  SNL_18650_NCA_25C_20-80_0.5-0.5C_d.pkl       NCA    25      20      1.979   
6  SNL_18650_NCA_25C_20-80_0.5-0.5C_d.pkl       NCA    25      40      1.979   
7  SNL_18650_NCA_25C_20-80_0.5-0.5C_d.pkl       NCA    25      60      1.979   
8  SNL_18650_NCA_25C_20-80_0.5-0.5C_d.pkl       NCA    25      80      1.979   
9  SNL_18650_NCA_25C_20-80_0.5-0.5C_d.pkl       NCA    25     100      1.979   

   retention     slope  life  log_life  
0   0.970457 -0.001437   331  2.519828  
1   0.956372 -0.000923   331

In [14]:
from sklearn.model_selection import GroupKFold, KFold, cross_val_predict
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X = pd.get_dummies(
    table[["budget", "cap_first", "retention", "slope", "temp", "chemistry"]],
    columns=["chemistry"]).astype(float)
y = table["log_life"].values
groups = table["cell"].values

models = {
    "Mean predictor": DummyRegressor(strategy="mean"),
    "Ridge (linear)": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Gradient boosting": GradientBoostingRegressor(random_state=0),
}
splits = {
    "Random row split (leaky)": KFold(5, shuffle=True, random_state=0),
    "Group split by cell": GroupKFold(5),
}

out = []
for sname, splitter in splits.items():
    for mname, model in models.items():
        pred = cross_val_predict(model, X, y, cv=splitter, groups=groups)
        err_log = np.abs(pred - y)
        err_cyc = np.abs(10**pred - 10**y)
        b100 = (table["budget"] == 100).values
        out.append({"split": sname, "model": mname,
                    "MAE_log10": err_log.mean().round(3),
                    "MAE_cycles": err_cyc.mean().round(0),
                    "MAE_cycles_budget100": err_cyc[b100].mean().round(0)})

print(pd.DataFrame(out).to_string(index=False))

                   split             model  MAE_log10  MAE_cycles  MAE_cycles_budget100
Random row split (leaky)    Mean predictor      0.299       611.0                 611.0
Random row split (leaky)    Ridge (linear)      0.091       147.0                 154.0
Random row split (leaky) Gradient boosting      0.030        43.0                  44.0
     Group split by cell    Mean predictor      0.298       610.0                 610.0
     Group split by cell    Ridge (linear)      0.098       159.0                 156.0
     Group split by cell Gradient boosting      0.080       120.0                 118.0
